# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:   
<br> Url: https://github.com/JavierRodriguezReina/03MIAR-Proyecto-Grupo16.git <br>
Problema:
> 1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

Los datos son:

- Número de actores 10
- Número de tomas 30
- Valores (nuestro github): [Enlace a los datos](https://github.com/JavierRodriguezReina/03MIAR-Proyecto-Grupo16/blob/main/Datos%20problema%20doblaje(30%20tomas%2C%2010%20actores).csv)

> 1 significa que el actor participa en la toma

> 0 significa que el actor no participa en la toma
                                        

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta

**Respuesta**

**Modelo del espacio de soluciones.** Una solución consiste en decidir **en qué día se graba cada toma**. La representamos, igual que en el modelo de tupla del árbol de expansión (p.ej. las N reinas), como una tupla de 30 posiciones:

$$(d_1, d_2, \dots, d_{30}), \qquad d_i = \text{día en que se graba la toma } i$$

donde cada componente puede tomar como valor cualquiera de los días disponibles. Como cada una de las 30 tomas puede ir a cualquier día, esto es una **variación con repetición** (combinatoria básica).

### Sin tener en cuenta las restricciones

Si hay $D$ días posibles, el número de tuplas distintas es:

$$VR_D^{30} = D^{30}$$

En el caso extremo cada toma va a su propio día, luego como máximo hacen falta **30 días** ($D = 30$):

$$D^{30} = 30^{30} \approx 2{,}06 \times 10^{44}$$

Este es el tamaño del árbol de expansión que recorrería la fuerza bruta (días etiquetados, sin descartar equivalentes ni días vacíos), y lo reutilizaremos al calcular su complejidad.

> **Nota:** si en lugar de tuplas contamos únicamente las **agrupaciones distintas** (sin etiquetar los días, sin repetir las equivalentes), el problema equivale a las *particiones de un conjunto* de 30 elementos, cuyo número es el **número de Bell** $B(30) \approx 8{,}47 \times 10^{23}$. Es un valor menor porque no sobrecuenta las ordenaciones de días equivalentes.

### Teniendo en cuenta todas las restricciones

La restricción del problema es **máximo 6 tomas por día**. Entonces ya no vale cualquier tupla: solo son válidas aquellas en las que **ningún valor de día aparece más de 6 veces** (ningún día concentra más de 6 tomas). Esto reduce el espacio a un subconjunto mucho menor del anterior, que calculamos por conteo en la siguiente celda.

In [ ]:
from functools import lru_cache
from math import comb

N_TOMAS = 30      # número de tomas
MAX_DIA = 6       # máximo de tomas por día (restricción)

# ----- SIN restricciones: variaciones con repetición (enfoque del temario) -----
# Cada toma se asigna a uno de los D días posibles -> D^N.
# En el caso extremo cada toma va a su propio día, así que D = N (30 días).
D = N_TOMAS
sin_restriccion = D ** N_TOMAS

# ----- Número de Bell B(N): agrupaciones DISTINTAS, sin etiquetar los días -----
# (particiones de un conjunto de N elementos). Se genera con el triángulo de Bell.
def bell(n):
    fila = [1]
    for _ in range(n):
        nueva = [fila[-1]]                 # cada fila empieza con el último de la anterior
        for x in fila:
            nueva.append(nueva[-1] + x)    # cada término = izquierda + el de arriba
        fila = nueva
    return fila[0]

bell_30 = bell(N_TOMAS)

# ----- CON restricciones (<= 6 tomas por día) -----
# Agrupaciones (particiones) donde ningún grupo/día supera MAX_DIA tomas.
# Fijamos una toma: su día tendrá tamaño s (1..MAX_DIA); elegimos sus s-1 compañeros
# de entre las N-1 restantes y particionamos el resto.
@lru_cache(None)
def particiones_restringidas(n, k=MAX_DIA):
    if n == 0:
        return 1
    return sum(comb(n - 1, s - 1) * particiones_restringidas(n - s, k)
               for s in range(1, min(k, n) + 1))

con_restriccion = particiones_restringidas(N_TOMAS, MAX_DIA)

# ----- Comparación -----
print(f"{'Enfoque':45}{'Valor':>12}   Cuenta")
print("-" * 90)
print(f"{'SIN restr. · variaciones con repetición D^N':45}{sin_restriccion:>12.3e}   {sin_restriccion}")
print(f"{'SIN restr. · número de Bell B(30)':45}{bell_30:>12.3e}   {bell_30}")
print(f"{'CON restr. · Bell restringido (<=6/día)':45}{con_restriccion:>12.3e}   {con_restriccion}")
print("-" * 90)
print(f"D^N es {sin_restriccion / bell_30:,.0f} veces mayor que B(30) "
      f"(sobrecuenta ordenaciones de días equivalentes y días vacíos).")
print(f"El Bell restringido es el {con_restriccion / bell_30:.1%} de B(30) "
      f"(la restricción <=6 elimina solo las particiones con grupos grandes).")

Enfoque                                             Valor   Cuenta
------------------------------------------------------------------------------------------
SIN restr. · variaciones con repetición D^N     2.059e+44   205891132094649000000000000000000000000000000
SIN restr. · número de Bell B(30)               8.467e+23   846749014511809332450147
CON restr. · Bell restringido (<=6/día)         7.264e+23   726391948970868949621309
------------------------------------------------------------------------------------------
D^N es 243,154,852,932,843,307,008 veces mayor que B(30) (sobrecuenta ordenaciones de días equivalentes y días vacíos).
El Bell restringido es el 85.8% de B(30) (la restricción <=6 elimina solo las particiones con grupos grandes).


Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

Una solución del problema consiste en **repartir las 30 tomas en días de grabación**. La cuestión es cómo representar ese reparto.

### Primera idea: una lista de conjuntos

Lo más intuitivo es guardar la solución como una **lista de días**, donde cada día es el **conjunto de tomas** que se graban en él:

$$[\;\{t_1, t_2, t_6\},\;\; \{t_3, t_5\},\;\; \{t_4\}\;]$$

Es una representación fiel: se lee directamente qué se graba cada día. Pero al pensar en cómo **recorrer el espacio de soluciones** aparecen dos problemas serios:

- **Longitud variable:** el número de días no es fijo, así que las soluciones no tienen todas la misma forma.
- **Invariante frágil:** cada toma debe aparecer **exactamente una vez** en toda la estructura. Cualquier operación de búsqueda (mover una toma, y sobre todo el **cruce** de un genético) rompe esa condición con facilidad y genera soluciones inválidas, con tomas duplicadas o perdidas, que habría que reparar a mano.

### Estructura elegida: una tupla de 30 enteros

Por eso cambiamos a una representación **posicional**: una tupla con una posición por toma, cuyo valor es el día que se le asigna.

$$(d_1, d_2, \dots, d_{30}), \qquad d_i = \text{día en que se graba la toma } i$$

Esta estructura resuelve los dos problemas anteriores y se adapta mucho mejor a la búsqueda:

- **Longitud fija (30 posiciones):** todas las soluciones tienen la misma forma.
- **Siempre válida por construcción:** cada toma ocupa una única posición, así que es imposible duplicarla o perderla. No hay nada que reparar.
- **Operadores triviales:** cambiar una decisión es cambiar un valor. En el **algoritmo genético** esta tupla es directamente el **cromosoma**: mutar es cambiar un gen y cruzar es partir el vector por un punto.
- **Permite contar el espacio de forma inmediata:** al ser una asignación posicional, el número de soluciones sale como variaciones con repetición, $D^{30}$ (pregunta anterior).

**Las dos representaciones son equivalentes** y se pasa de una a otra agrupando las tomas que comparten día (*decodificar*). Usamos la tupla para **buscar** y la agrupación en días solo cuando hace falta **evaluar** el coste de la solución:

$$\underbrace{(0,\,0,\,1,\,2,\,1,\,0)}_{\text{tupla: se busca}} \;\xrightarrow{\;\text{decodificar}\;}\; \underbrace{[\,\{t_1,t_2,t_6\},\;\{t_3,t_5\},\;\{t_4\}\,]}_{\text{días: se evalúa}}$$

**Conclusión:** el espacio de soluciones se modela con una **tupla de 30 enteros**. Aunque la lista de conjuntos es más legible, la tupla es la que mejor se adapta al problema porque tiene longitud fija, es válida por construcción y permite aplicar los operadores de búsqueda (cruce y mutación) sin generar soluciones inconsistentes.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta